# EON GPU Node Runner — Colab (Tor client)

Joins the **EON sovereign mesh as `gpu-node1`** over its **Tor onion door** and trains the SNN with **real torch on this Colab GPU**, then posts trained weights back so the mesh bumps `model:active_version` and mirrors `state/models/<ver>.json`.

**How it reaches the mesh:** the mesh binds `127.0.0.1:8787` only (zero public exposure) and is exposed to the internet as a **Tor hidden service**: `o3izfmjjt2pmsgauio7fau3ykiwm5ion4ltojv7zegdpp7n74tfqsqad.onion:80`. Cell 2 installs + boots a Tor client, Cell 3 uses a **pure-stdlib raw SOCKS5 client** (no pip deps) so every request is Tor-routed (onion-safe DNS).

- **Auth:** send `Authorization: Bearer <EON_TOKEN>`. The mesh is token-gated for all mutating routes (`/api/ml/run`, `/api/ml/complete`, `/api/ml/version`, `/api/nodes`, ...) — GET reads are open.
- **No token? No mesh?** The notebook auto-falls back to an offline **dry-run self-test** so you can still validate the training flow.

Cells: 1) config 2) boot Tor 3) SOCKS5 client + fetch job 4) install torch 5) train & emit weights 6) post back + verify.


## 1. Config — paste your token

In [ ]:
import json, os, sys, socket, time, subprocess, urllib.request

ONION = os.environ.get("EON_ONION", "o3izfmjjt2pmsgauio7fau3ykiwm5ion4ltojv7zegdpp7n74tfqsqad.onion")
EON_TOKEN = os.environ.get("EON_TOKEN", "")   # <-- paste mesh token here if set
NODE_ID = os.environ.get("EON_NODE_ID", "gpu-node1")
SOCKS_PORT = int(os.environ.get("SOCKS_PORT", "9050"))
COLAB = 'google.colab' in sys.modules
print('Colab:', COLAB, '| node:', NODE_ID, '| onion:', ONION, '| token set:', bool(EON_TOKEN))


## 2. Install + boot a Tor client (for the onion gateway)

In [ ]:
# In Colab, install Tor and start a SOCKS5 proxy on 127.0.0.1:9050 in the background.
if COLAB:
    subprocess.run(['apt-get', 'install', '-y', '-q', 'tor'], check=False, capture_output=True)
    subprocess.run(['nohup', 'tor', '--SocksPort', str(SOCKS_PORT), '>', '/tmp/tor.log', '2>&1', '&'], shell=True)
    for _ in range(60):
        try:
            log = open('/tmp/tor.log').read()
            if 'Bootstrapped 100%' in log: break
        except Exception: pass
        time.sleep(2)
    try:
        log = open('/tmp/tor.log').read()
        print('tor:', [l for l in log.splitlines() if 'Bootstrapped' in l][-1])
    except Exception as e:
        print('tor log unavailable:', repr(e)[:80])
else:
    print('not Colab; assuming a local tor on 127.0.0.1:%d' % SOCKS_PORT)


## 3. Pure-stdlib raw SOCKS5 client + fetch the next job

In [ ]:
# Minimal SOCKS5h client (raw sockets, no pip deps) — copy of twin_sync.py pattern.
def socks_connect(host, port, timeout=60):
    s = socket.create_connection(('127.0.0.1', SOCKS_PORT), timeout)
    s.sendall(b'\x05\x01\x00')  # SOCKS5, 1 method, no-auth
    r = s.recv(2)
    if r != b'\x05\x00': s.close(); raise IOError('socks handshake failed')
    hb = host.encode()
    s.sendall(b'\x05\x01\x00\x03' + bytes([len(hb)]) + hb + (port).to_bytes(2, 'big'))
    r = s.recv(10)
    if len(r) < 2 or r[1] != 0: s.close(); raise IOError('socks connect failed')
    return s

def _dechunk(raw):
    out = b''
    i = 0
    try:
        while i < len(raw):
            j = raw.index(b'\r\n', i)
            size = int(raw[i:j], 16)
            i = j + 2
            if size == 0: break
            out += raw[i:i + size]
            i += size + 2
        return out
    except Exception:
        return raw

def tor_req(method, path, body=None, timeout=90):
    s = socks_connect(ONION, 80, timeout)
    payload = body.encode() if body else b''
    hdrs = {'Host': ONION, 'Content-Type': 'application/json',
            'Content-Length': str(len(payload)), 'Connection': 'close'}
    if EON_TOKEN: hdrs['Authorization'] = 'Bearer ' + EON_TOKEN
    req = (method + ' ' + path + ' HTTP/1.1\r\n' +
           ''.join('%s: %s\r\n' % (k, v) for k, v in hdrs.items()) + '\r\n')
    s.sendall(req.encode() + payload)
    resp = b''
    while True:
        c = s.recv(65536)
        if not c: break
        resp += c
    s.close()
    head, _, b = resp.partition(b'\r\n\r\n')
    status = int(head.split(b' ')[1])
    return status, json.loads(_dechunk(b).decode('utf-8') or '{}')

def tor_get(path): return tor_req('GET', path)
def tor_post(path, payload): return tor_req('POST', path, json.dumps(payload))

# Reachability + node registration
reachable = False
try:
    st, h = tor_get('/api/health')
    print('mesh via Tor:', st, h.get('service'))
    reachable = True
except Exception as e:
    print('mesh unreachable:', repr(e)[:120])

if reachable:
    try:
        st, b = tor_post('/api/nodes', {'node_id': NODE_ID, 'name': NODE_ID,
                                        'type': 'cloud-gpu',
                                        'capabilities': ['compute', 'gpu', 'training']})
        print('registered', NODE_ID, 'http', st)
    except Exception as e:
        print('register error:', repr(e)[:120])

JOB_JSON = os.environ.get('JOB_JSON', '')
if JOB_JSON:
    job = json.loads(JOB_JSON)
elif reachable:
    st, job = tor_get('/api/ml/job/latest')
    print('pull job http', st)
else:
    print('gateway unreachable -> inline dry-run job')
    job = {'task_id': 'ml-dryrun-colab', 'framework': 'snn', 'gpu': True,
           'code': '# colab dry-run', 'data': {'n': 128}}

if not job or 'task_id' not in job:
    print('no runnable job right now (queue empty). Re-run later.')
else:
    TASK_ID = job['task_id']
    print('task:', TASK_ID, '| framework:', job.get('framework'), '| gpu:', job.get('gpu'))


## 4. Install the torch stack (real GPU training)

In [ ]:
subprocess.run(['pip', 'install', '--quiet', 'torch', 'torchvision', 'snntorch'])
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARN: no CUDA on this box (dry-run/CPU training)')


## 5. Train the SNN for real and emit weights

In [ ]:
import math, random

def train_real(task_id, epochs=3, samples=128):
    import snntorch as snn
    from snntorch import spikegen
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    torch.manual_seed(abs(hash(task_id)) % (2**31))
    net = nn.Sequential(
        nn.Linear(28 * 28, 64),
        snn.Leaky(beta=0.9, spike_grad=snn.surrogate.fast_sigmoid(), init_hidden=True),
        nn.Linear(64, 10),
        snn.Leaky(beta=0.9, spike_grad=snn.surrogate.fast_sigmoid(), init_hidden=True, output=True),
    )
    X = torch.randn(samples, 28 * 28)
    y = torch.randint(0, 10, (samples,))
    loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    lossf = nn.CrossEntropyLoss()
    acc = 0.0
    for ep in range(epochs):
        correct = total = 0
        for xb, yb in loader:
            for l in net:
                if isinstance(l, snn.Leaky): l.reset_hidden()
            spk, mem = net(xb)
            loss = lossf(mem, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            pred = mem.argmax(dim=1)
            correct += (pred == yb).sum().item(); total += yb.numel()
        acc = correct / total
        print(f'  epoch {ep+1}/{epochs} loss={loss.item():.4f} acc={acc:.3f}')
    ws = []
    for p in net.parameters():
        ws.extend(p.detach().cpu().reshape(-1).tolist())
    return {'metrics': {'accuracy': round(acc, 4), 'epochs': epochs, 'samples': samples,
                        'cuda': torch.cuda.is_available()},
            'weights': ws, 'shape': {'total': len(ws)},
            'version': 'snn-colab-%d' % int(time.time() * 1000)}

def train_dryrun(task_id):
    rng = random.Random(abs(hash(task_id)) % (2**31))
    return {'metrics': {'accuracy': round(0.5 + rng.random() * 0.49, 4), 'epochs': 3, 'samples': 128,
                        'cuda': False, 'dryrun': True},
            'weights': [round(rng.random() * 2 - 1, 6) for _ in range(64)],
            'shape': {'total': 64}, 'version': 'snn-sim-%d' % int(time.time() * 1000)}

try:
    RESULT = train_real(TASK_ID)
    PROVIDER = 'colab-gpu'
except Exception as e:
    print('real train failed -> dryrun:', repr(e)[:120])
    RESULT = train_dryrun(TASK_ID)
    PROVIDER = 'colab-dryrun'
print('provider:', PROVIDER, '| weights:', RESULT['shape']['total'], '| version:', RESULT['version'])


## 6. Post weights back + verify version bump

In [ ]:
if ('RESULT' in dir()) and reachable:
    st, body = tor_post('/api/ml/complete',
                        {'task_id': TASK_ID, 'status': 'done', 'result': RESULT, 'provider': PROVIDER})
    print('complete http', st, body.get('status'))
    st, ver = tor_get('/api/ml/version')
    print('active_version now:', ver)
    print('EON GPU NODE DONE — weights published over Tor. provider=%s' % PROVIDER)
else:
    print('No mesh reach (dry-run validated locally). Saved /content/eon_result.json')
    with open('/content/eon_result.json', 'w') as f:
        json.dump(RESULT, f)
